In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from pathlib import Path
from scipy.stats import shapiro


# ============================================================
# 1. APHC results location
# ============================================================

ROOT = Path("Results") / "APHC"

# If needed, replace ROOT with the full path to your APHC folder.

print("Reading results from:", ROOT.resolve(), flush=True)

settings_files = sorted(ROOT.glob("*/*/Settings.csv"))

if not settings_files:
    raise FileNotFoundError(
        f"No Settings.csv files found under {ROOT.resolve()}. "
        "Check the ROOT path."
    )

print(f"Found {len(settings_files)} scenarios.", flush=True)


# ============================================================
# 2. Helper functions
# ============================================================

def make_predictors(frame):
    """Create intercept, inverse temperature, and log voltage."""
    temperature = frame["Temperature"].to_numpy(dtype=float)
    voltage = frame["Voltage"].to_numpy(dtype=float)

    if (
        not np.isfinite(temperature).all()
        or not np.isfinite(voltage).all()
        or np.any(temperature <= 0)
        or np.any(voltage <= 0)
    ):
        raise ValueError("Temperature and voltage must be positive and finite.")

    X = pd.DataFrame({
        "const": np.ones(len(frame)),
        "inv_temp": 1.0 / temperature,
        "log_vol": np.log(voltage),
    })

    if np.linalg.matrix_rank(X.to_numpy()) != 3:
        raise ValueError("Regression predictors are not full rank.")

    return X


def fit_regression(y, X):
    """OLS estimates with approximate HC3/t confidence intervals."""
    y = np.asarray(y, dtype=float)

    if len(y) <= 3 or not np.isfinite(y).all():
        raise ValueError("Insufficient or invalid regression observations.")

    return sm.OLS(y, X).fit(
        cov_type="HC3",
        use_t=True,
    )


def coefficient_records(
    model,
    names,
    settings,
    metadata,
    failure_message="",
):
    """Return one record for each regression coefficient."""
    estimates = np.full(3, np.nan)
    ci95 = np.full((3, 2), np.nan)
    ci99 = np.full((3, 2), np.nan)

    fit_success = False
    ci_success = False
    message = failure_message

    if model is not None:
        estimates = np.asarray(model.params, dtype=float)
        fit_success = bool(np.isfinite(estimates).all())

        if fit_success:
            try:
                ci95 = np.asarray(
                    model.conf_int(alpha=0.05), dtype=float
                )
                ci99 = np.asarray(
                    model.conf_int(alpha=0.01), dtype=float
                )

                ci_success = bool(
                    np.isfinite(ci95).all()
                    and np.isfinite(ci99).all()
                    and np.all(ci95[:, 0] <= ci95[:, 1])
                    and np.all(ci99[:, 0] <= ci99[:, 1])
                )

                if not ci_success:
                    message = "Invalid confidence intervals."

            except (ValueError, np.linalg.LinAlgError) as error:
                message = str(error)
        else:
            message = "Nonfinite regression coefficients."

    rows = []

    for k, name in enumerate(names):
        true_value = float(settings[name])

        rows.append({
            **metadata,
            "Coefficient": name,
            "True_value": true_value,
            "Estimate": estimates[k] if fit_success else np.nan,
            "Error": (
                estimates[k] - true_value
                if fit_success else np.nan
            ),
            "CI95_lower": ci95[k, 0] if ci_success else np.nan,
            "CI95_upper": ci95[k, 1] if ci_success else np.nan,
            "CI99_lower": ci99[k, 0] if ci_success else np.nan,
            "CI99_upper": ci99[k, 1] if ci_success else np.nan,
            "Fit_success": fit_success,
            "CI_success": ci_success,
            "Message": message,
            "CI_method": "OLS HC3 with t critical values",
        })

    return rows


# ============================================================
# 3. Process all parameter settings and datasets
# ============================================================

all_coefficient_summaries = []
all_mean_regressions = []
all_normality_results = []

for scenario_number, settings_path in enumerate(
    settings_files, start=1
):
    scenario_dir = settings_path.parent
    parameter_name = scenario_dir.parent.name
    dataset_name = scenario_dir.name

    settings_df = pd.read_csv(settings_path)

    if len(settings_df) != 1:
        raise ValueError(
            f"Expected one settings row: {settings_path}"
        )

    settings = settings_df.iloc[0]

    if (
        str(settings["Dataset"]) != dataset_name
        or str(settings["Parameter_setting"]) != parameter_name
    ):
        raise ValueError(
            f"Folder/settings mismatch: {settings_path}"
        )

    n_pairs = int(settings["N_pairs"])
    n_sim = int(settings["N_simulations"])

    if n_pairs <= 3 or n_sim < 1:
        raise ValueError(
            f"Invalid simulation dimensions: {settings_path}"
        )

    metadata = {
        "Parameter_setting": parameter_name,
        "Dataset": dataset_name,
        "n": int(settings["n"]),
        "m": int(settings["m"]),
        "Planned_observed_fraction": (
            float(settings["m"]) / float(settings["n"])
        ),
        "t": float(settings["t"]),
    }

    print(
        f"\nScenario {scenario_number}/{len(settings_files)}: "
        f"{parameter_name} | {dataset_name}",
        flush=True,
    )

    regression_dir = scenario_dir / "Regression"
    regression_dir.mkdir(exist_ok=True)

    # --------------------------------------------------------
    # 3A. Read and validate simulation results
    # --------------------------------------------------------

    summary_path = scenario_dir / "True_vals_&_MLE.csv"
    mean_df = pd.read_csv(summary_path)

    expected_pairs = set(range(1, n_pairs + 1))
    expected_repetitions = set(range(1, n_sim + 1))

    if (
        len(mean_df) != n_pairs
        or mean_df["Pair"].duplicated().any()
        or set(mean_df["Pair"]) != expected_pairs
    ):
        raise ValueError(
            f"Incomplete or duplicate stress pairs: {summary_path}"
        )

    if (
        not (mean_df["Dataset"] == dataset_name).all()
        or not (
            mean_df["Parameter_setting"] == parameter_name
        ).all()
    ):
        raise ValueError(
            f"Scenario mismatch in {summary_path}"
        )

    mean_df = mean_df.sort_values("Pair").reset_index(drop=True)

    frames = []

    for pair in range(1, n_pairs + 1):
        file_path = (
            scenario_dir / "Replicates" / f"pair_{pair}.csv"
        )

        if not file_path.exists():
            raise FileNotFoundError(
                f"Missing repetition file: {file_path}"
            )

        frame = pd.read_csv(file_path)

        if (
            len(frame) != n_sim
            or frame["Repetition"].duplicated().any()
            or set(frame["Repetition"]) != expected_repetitions
            or not (frame["Pair"] == pair).all()
            or not (frame["Dataset"] == dataset_name).all()
            or not (
                frame["Parameter_setting"] == parameter_name
            ).all()
        ):
            raise ValueError(
                f"Inconsistent repetition data: {file_path}"
            )

        flags = (
            frame["Fit_success"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        if not flags.isin(["true", "false"]).all():
            raise ValueError(
                f"Invalid Fit_success values: {file_path}"
            )

        frame["Fit_success"] = flags.eq("true")

        reference = mean_df.loc[
            mean_df["Pair"] == pair
        ].iloc[0]

        for column in ("Temperature", "Voltage"):
            if not np.allclose(
                frame[column].to_numpy(dtype=float),
                float(reference[column]),
                rtol=1e-10,
                atol=1e-12,
            ):
                raise ValueError(
                    f"{column} mismatch: {file_path}"
                )

        frames.append(frame)

    repetitions_df = pd.concat(frames, ignore_index=True)

    X = make_predictors(mean_df)

    # --------------------------------------------------------
    # 3B. Original regressions on average estimates
    # --------------------------------------------------------

    mean_rows = []
    normality_rows = []

    for parameter, response_column, names in (
        ("Alpha", "Alpha MLE", ["c0", "c1", "c2"]),
        ("Lambda", "Lambda MLE", ["a0", "a1", "a2"]),
    ):
        model = None
        message = ""

        try:
            model = fit_regression(
                mean_df[response_column], X
            )

        except (ValueError, np.linalg.LinAlgError) as error:
            message = str(error)

        mean_rows.extend(
            coefficient_records(
                model,
                names,
                settings,
                {
                    **metadata,
                    "Response": parameter,
                    "Analysis": "Regression on mean estimates",
                },
                message,
            )
        )

        shapiro_stat = np.nan
        shapiro_p = np.nan
        normality_message = message

        if model is not None:
            residuals = np.asarray(
                model.resid, dtype=float
            )

            # Save residual Q-Q plot.
            qq_dir = scenario_dir / "QQ" / parameter
            qq_dir.mkdir(parents=True, exist_ok=True)

            fig = sm.qqplot(residuals, line="s")
            fig.axes[0].set_title(
                f"{parameter}: residuals from regression on means"
            )
            fig.savefig(
                qq_dir / "after_regression.png",
                bbox_inches="tight",
            )
            plt.close(fig)

            # Save regression summary.
            with open(
                regression_dir / f"Mean_regression_{parameter}.txt",
                "w",
                encoding="utf-8",
            ) as file:
                file.write(model.summary().as_text())

            # Shapiro-Wilk test.
            if (
                len(residuals) >= 3
                and np.isfinite(residuals).all()
                and np.ptp(residuals) > 0
            ):
                shapiro_stat, shapiro_p = shapiro(residuals)
                normality_message = ""
            else:
                normality_message = (
                    "Shapiro test unavailable: invalid, constant, "
                    "or insufficient residuals."
                )

        normality_rows.append({
            **metadata,
            "Response": parameter,
            "Analysis": "Residuals from regression on mean estimates",
            "Shapiro_statistic": shapiro_stat,
            "Shapiro_p_value": shapiro_p,
            "Message": normality_message,
        })

    pd.DataFrame(mean_rows).to_csv(
        regression_dir / "Mean_regression_coefficients.csv",
        index=False,
    )

    pd.DataFrame(normality_rows).to_csv(
        regression_dir / "Residual_normality_tests.csv",
        index=False,
    )

    all_mean_regressions.extend(mean_rows)
    all_normality_results.extend(normality_rows)

    # --------------------------------------------------------
    # 3C. Regression for every simulation repetition
    # --------------------------------------------------------

    coefficient_rows = []

    for repetition, group in repetitions_df.groupby(
        "Repetition", sort=True
    ):
        group = group.sort_values("Pair").reset_index(drop=True)

        # Require the full set of successful stress-pair fits.
        # Do not silently fit a smaller regression.
        complete_fit = (
            len(group) == n_pairs
            and set(group["Pair"]) == expected_pairs
            and group["Fit_success"].all()
            and np.isfinite(
                group[["Alpha_est", "Lambda_est"]]
                .to_numpy(dtype=float)
            ).all()
        )

        for parameter, response_column, names in (
            ("Alpha", "Alpha_est", ["c0", "c1", "c2"]),
            ("Lambda", "Lambda_est", ["a0", "a1", "a2"]),
        ):
            model = None
            message = ""

            if not complete_fit:
                message = (
                    "At least one stress pair has a failed "
                    "or invalid first-stage fit."
                )
            else:
                try:
                    model = fit_regression(
                        group[response_column], X
                    )

                except (
                    ValueError,
                    np.linalg.LinAlgError,
                ) as error:
                    message = str(error)

            coefficient_rows.extend(
                coefficient_records(
                    model,
                    names,
                    settings,
                    {
                        **metadata,
                        "Response": parameter,
                        "Repetition": int(repetition),
                    },
                    message,
                )
            )

    coefficient_df = pd.DataFrame(coefficient_rows)

    coefficient_df.to_csv(
        regression_dir / "Coefficient_estimates_by_repetition.csv",
        index=False,
    )

    # --------------------------------------------------------
    # 3D. Bias, SD, RMSE and confidence-interval coverage
    # --------------------------------------------------------

    scenario_summaries = []

    for name in ("a0", "a1", "a2", "c0", "c1", "c2"):
        group = coefficient_df.loc[
            coefficient_df["Coefficient"] == name
        ].copy()

        true_value = float(settings[name])

        fit_valid = group["Fit_success"].to_numpy(dtype=bool)
        ci_valid = group["CI_success"].to_numpy(dtype=bool)

        estimates = group["Estimate"].to_numpy(dtype=float)
        valid_estimates = estimates[fit_valid]
        errors = valid_estimates - true_value

        summary = {
            **metadata,
            "Coefficient": name,
            "True_value": true_value,
            "N_repetitions": n_sim,
            "N_valid_fits": int(fit_valid.sum()),
            "N_valid_CIs": int(ci_valid.sum()),
            "Fit_failure_rate": 1 - fit_valid.mean(),
            "CI_failure_rate": 1 - ci_valid.mean(),
            "Mean_estimate": (
                valid_estimates.mean()
                if errors.size else np.nan
            ),
            "Bias": (
                errors.mean()
                if errors.size else np.nan
            ),
            "SD": (
                valid_estimates.std(ddof=0)
                if errors.size else np.nan
            ),
            "RMSE": (
                np.sqrt(np.mean(errors ** 2))
                if errors.size else np.nan
            ),
            "CI_method": "OLS HC3 with t critical values",
        }

        for level in (95, 99):
            lower = group[
                f"CI{level}_lower"
            ].to_numpy(dtype=float)

            upper = group[
                f"CI{level}_upper"
            ].to_numpy(dtype=float)

            covered = (
                ci_valid
                & (lower <= true_value)
                & (true_value <= upper)
            )

            # Coverage conditional on obtaining a valid CI.
            summary[f"Coverage{level}_valid"] = (
                covered[ci_valid].mean()
                if ci_valid.any() else np.nan
            )

            summary[f"Mean_width{level}"] = (
                (upper[ci_valid] - lower[ci_valid]).mean()
                if ci_valid.any() else np.nan
            )

            # Covering intervals divided by all attempted repetitions.
            summary[f"Covering_CI{level}_all"] = covered.mean()

        scenario_summaries.append(summary)

    scenario_summary_df = pd.DataFrame(scenario_summaries)

    scenario_summary_df.to_csv(
        regression_dir / "Coefficient_summary.csv",
        index=False,
    )

    all_coefficient_summaries.extend(scenario_summaries)

    # --------------------------------------------------------
    # 3E. Save combined summaries after every scenario
    # --------------------------------------------------------

    combined = pd.DataFrame(all_coefficient_summaries)

    combined.to_csv(
        ROOT / "All_regression_coefficient_summaries.csv",
        index=False,
    )

    combined.loc[
        combined["Parameter_setting"] == parameter_name
    ].to_csv(
        scenario_dir.parent / "Regression_coefficient_summary.csv",
        index=False,
    )

    pd.DataFrame(all_mean_regressions).to_csv(
        ROOT / "All_mean_regression_coefficients.csv",
        index=False,
    )

    pd.DataFrame(all_normality_results).to_csv(
        ROOT / "All_residual_normality_tests.csv",
        index=False,
    )

    min_valid_fits = scenario_summary_df["N_valid_fits"].min()
    min_valid_cis = scenario_summary_df["N_valid_CIs"].min()

    print(
        f"Completed: {parameter_name} | {dataset_name}\n"
        f"Minimum valid fits across coefficients: "
        f"{min_valid_fits}/{n_sim}\n"
        f"Minimum valid CIs across coefficients: "
        f"{min_valid_cis}/{n_sim}\n"
        f"Saved to: {regression_dir}",
        flush=True,
    )


# ============================================================
# 4. Finished
# ============================================================

print(
    f"\nCompleted all {len(settings_files)} APHC regression scenarios.",
    flush=True,
)

print(
    "Combined coefficient summary:",
    (ROOT / "All_regression_coefficient_summaries.csv").resolve(),
    flush=True,
)

Reading results from: C:\Users\NetComp\Documents\RJ\Results\APHC
Found 200 scenarios.

Scenario 1/200: baseline | m100
Completed: baseline | m100
Minimum valid fits across coefficients: 1000/1000
Minimum valid CIs across coefficients: 1000/1000
Saved to: Results\APHC\baseline\m100\Regression

Scenario 2/200: baseline | m100_p25
Completed: baseline | m100_p25
Minimum valid fits across coefficients: 1000/1000
Minimum valid CIs across coefficients: 1000/1000
Saved to: Results\APHC\baseline\m100_p25\Regression

Scenario 3/200: baseline | m100n
Completed: baseline | m100n
Minimum valid fits across coefficients: 1000/1000
Minimum valid CIs across coefficients: 1000/1000
Saved to: Results\APHC\baseline\m100n\Regression

Scenario 4/200: baseline | m100nm
Completed: baseline | m100nm
Minimum valid fits across coefficients: 1000/1000
Minimum valid CIs across coefficients: 1000/1000
Saved to: Results\APHC\baseline\m100nm\Regression

Scenario 5/200: baseline | m120_n180_balanced
Completed: baselin